# 02 RAG Starter Kit - ContribGuide AI

This notebook runs the complete ContribGuide AI RAG pipeline on Google Colab with GPU acceleration.
It connects to your working project and can serve as the backend for the frontend at http://localhost:5174

## Features:
- Full RAG pipeline (load → chunk → embed → store → retrieve → generate)
- Google Generative AI (Gemma) integration
- ChromaDB vector store with persistence
- BM25 keyword search + semantic search hybrid retrieval
- ngrok tunnel for frontend connectivity
- Document upload and processing
- Chat API with citations

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install -q fastapi uvicorn pydantic pydantic-settings python-multipart pymupdf markdown sentence-transformers chromadb rank-bm25 google-generativeai httpx nest-asyncio pyngrok tqdm

In [ ]:
# ============================================================
# CELL 2: Clone Project & Setup Environment
# ============================================================
import os
import sys

# Clone the repository (replace with your fork URL)
REPO_URL = "https://github.com/YOUR_USERNAME/contribguide-ai.git"  # <-- CHANGE THIS
!git clone $REPO_URL /content/contribguide-ai 2>/dev/null || echo "Repo already cloned or error"

%cd /content/contribguide-ai/backend

# Add to Python path
sys.path.insert(0, '/content/contribguide-ai/backend')

# Verify structure
!ls -la

In [ ]:
# ============================================================
# CELL 3: Configure Environment Variables
# ============================================================
import os

# ===== REPLACE WITH YOUR ACTUAL VALUES =====
GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY_HERE"  # <-- CHANGE THIS

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GEMMA_MODEL"] = "gemma-4-26b-a4b-it"
os.environ["EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"
os.environ["CHROMA_PERSIST_DIRECTORY"] = "/content/chroma"
os.environ["UPLOAD_DIRECTORY"] = "/content/uploads"
os.environ["CHUNK_SIZE"] = "800"
os.environ["CHUNK_OVERLAP"] = "150"
os.environ["TOP_K"] = "3"
os.environ["RELEVANCE_THRESHOLD"] = "0.25"
os.environ["ENABLE_RERANKER"] = "false"
os.environ["FRONTEND_ORIGIN"] = "*"
os.environ["MAX_UPLOAD_MB"] = "20"

# Create directories
os.makedirs("/content/chroma", exist_ok=True)
os.makedirs("/content/uploads", exist_ok=True)

print("Environment configured:")
print(f"  Model: {os.environ['GEMMA_MODEL']}")
print(f"  Embedding: {os.environ['EMBEDDING_MODEL']}")
print(f"  Chroma: {os.environ['CHROMA_PERSIST_DIRECTORY']}")
print(f"  Uploads: {os.environ['UPLOAD_DIRECTORY']}")

In [ ]:
# ============================================================
# CELL 4: Verify Google AI Connection & Model Availability
# ============================================================
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)

print("Available Gemma models:")
gemma_models = []
for m in genai.list_models():
    if "gemma" in m.name.lower():
        gemma_models.append(m.name)
        print(f"  ✓ {m.name}")

if not gemma_models:
    print("  ⚠ No Gemma models found. Check API key and region.")
else:
    print(f"\nTotal: {len(gemma_models)} Gemma models available")

In [ ]:
# ============================================================
# CELL 5: Initialize RAG Pipeline Components
# ============================================================
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings as ChromaSettings
import uuid
import json

# Initialize embedding model
print("Loading embedding model...")
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"  Embedding dimension: {embedder.get_sentence_embedding_dimension()}")

# Initialize ChromaDB with persistence
print("Initializing ChromaDB...")
chroma_client = chromadb.PersistentClient(
    path="/content/chroma",
    settings=ChromaSettings(anonymized_telemetry=False)
)

# Get or create collection
collection = chroma_client.get_or_create_collection(
    name="contribguide_docs",
    metadata={"hnsw:space": "cosine"}
)
print(f"  Collection: {collection.name} (count: {collection.count()})")

# Initialize BM25 for keyword search
from rank_bm25 import BM25Okapi
bm25_corpus = []
bm25_doc_ids = []
bm25_index = None

def rebuild_bm25():
    global bm25_corpus, bm25_doc_ids, bm25_index
    all_docs = collection.get(include=["documents", "metadatas"])
    bm25_corpus = [doc.split() for doc in all_docs["documents"]]
    bm25_doc_ids = all_docs["ids"]
    if bm25_corpus:
        bm25_index = BM25Okapi(bm25_corpus)
        print(f"  BM25 index rebuilt with {len(bm25_corpus)} documents")
    else:
        bm25_index = None
        print("  BM25 index empty")

rebuild_bm25()

In [ ]:
# ============================================================
# CELL 6: Document Processing Functions
# ============================================================
import fitz  # pymupdf
import markdown
import re

def load_document(filepath: str):
    """Load document and return list of pages with text and page numbers."""
    ext = os.path.splitext(filepath)[1].lower()
    pages = []
    
    if ext == ".pdf":
        doc = fitz.open(filepath)
        for i, page in enumerate(doc):
            text = page.get_text()
            if text.strip():
                pages.append({"text": text, "page": i + 1})
        doc.close()
    elif ext in (".md", ".markdown"):
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        html = markdown.markdown(content)
        # Simple text extraction from HTML
        text = re.sub(r'<[^>]+>', '', html)
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            pages.append({"text": text, "page": 1})
    elif ext == ".txt":
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read().strip()
        if text:
            pages.append({"text": text, "page": 1})
    
    return pages

def clean_text(text: str) -> str:
    """Clean and normalize text."""
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

def chunk_document(text: str, source: str, doc_type: str, chunk_size: int = 800, chunk_overlap: int = 150, page: int = 1):
    """Split document into overlapping chunks."""
    chunks = []
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = min(start + chunk_size, text_len)
        
        # Try to break at sentence boundary
        if end < text_len:
            last_period = text.rfind('.', start, end)
            if last_period > start + chunk_size // 2:
                end = last_period + 1
        
        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append({
                "text": chunk_text,
                "source": source,
                "doc_type": doc_type,
                "page": page,
                "chunk_id": str(uuid.uuid4())
            })
        
        start = end - chunk_overlap
        if start >= text_len:
            break
    
    return chunks

In [ ]:
# ============================================================
# CELL 7: Upload & Process Documents
# ============================================================
def process_and_store_document(filepath: str, filename: str):
    """Load, chunk, embed, and store a document."""
    ext = os.path.splitext(filename)[1].lower()
    
    print(f"Processing: {filename}")
    
    # Load document
    pages = load_document(filepath)
    print(f"  Loaded {len(pages)} pages")
    
    # Chunk all pages
    all_chunks = []
    for page in pages:
        chunks = chunk_document(
            clean_text(page["text"]),
            filename,
            ext.lstrip("."),
            chunk_size=800,
            chunk_overlap=150,
            page=page["page"]
        )
        all_chunks.extend(chunks)
    
    print(f"  Created {len(all_chunks)} chunks")
    
    if not all_chunks:
        print("  No content to index")
        return {"filename": filename, "chunks": 0}
    
    # Delete existing chunks for this source
    try:
        existing = collection.get(where={"source": filename})
        if existing["ids"]:
            collection.delete(ids=existing["ids"])
            print(f"  Deleted {len(existing['ids'])} old chunks")
    except:
        pass
    
    # Generate embeddings
    print("  Generating embeddings...")
    texts = [c["text"] for c in all_chunks]
    embeddings = embedder.encode(texts, show_progress_bar=True).tolist()
    
    # Store in ChromaDB
    ids = [c["chunk_id"] for c in all_chunks]
    metadatas = [
        {
            "source": c["source"],
            "doc_type": c["doc_type"],
            "page": c["page"]
        }
        for c in all_chunks
    ]
    
    collection.add(
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    
    # Rebuild BM25 index
    rebuild_bm25()
    
    print(f"  ✓ Stored {len(all_chunks)} chunks")
    return {"filename": filename, "chunks": len(all_chunks), "total_chunks": collection.count()}

# Test with demo files if they exist
demo_files = [
    "/content/contribguide-ai/demo-contribution-guide.md",
    "/content/contribguide-ai/demo-contribution-guide.txt",
]

for demo_file in demo_files:
    if os.path.exists(demo_file):
        result = process_and_store_document(demo_file, os.path.basename(demo_file))
        print(f"Result: {result}")
    else:
        print(f"Demo file not found: {demo_file}")

In [ ]:
# ============================================================
# CELL 8: Hybrid Retrieval (Semantic + Keyword)
# ============================================================
def hybrid_retrieve(query: str, top_k: int = 5, alpha: float = 0.5):
    """Hybrid retrieval combining semantic and keyword search."""
    results = {"documents": [], "metadatas": [], "distances": [], "ids": []}
    
    # Semantic search
    query_embedding = embedder.encode([query]).tolist()
    semantic_results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k * 2,  # Get more for reranking
        include=["documents", "metadatas", "distances"]
    )
    
    semantic_docs = semantic_results["documents"][0] if semantic_results["documents"] else []
    semantic_metas = semantic_results["metadatas"][0] if semantic_results["metadatas"] else []
    semantic_dists = semantic_results["distances"][0] if semantic_results["distances"] else []
    semantic_ids = semantic_results["ids"][0] if semantic_results["ids"] else []
    
    # Keyword search (BM25)
    keyword_docs = []
    keyword_metas = []
    keyword_scores = []
    keyword_ids = []
    
    if bm25_index and bm25_corpus:
        tokenized_query = query.split()
        bm25_scores = bm25_index.get_scores(tokenized_query)
        top_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:top_k * 2]
        
        for idx in top_indices:
            if bm25_scores[idx] > 0:
                keyword_docs.append(bm25_corpus[idx])
                keyword_ids.append(bm25_doc_ids[idx])
                keyword_scores.append(bm25_scores[idx])
    
    # Combine and deduplicate
    combined = {}
    
    # Add semantic results
    for i, doc_id in enumerate(semantic_ids):
        score = 1 - semantic_dists[i]  # Convert distance to similarity
        combined[doc_id] = {
            "document": semantic_docs[i],
            "metadata": semantic_metas[i],
            "semantic_score": score,
            "keyword_score": 0
        }
    
    # Add keyword results
    for i, doc_id in enumerate(keyword_ids):
        if doc_id in combined:
            combined[doc_id]["keyword_score"] = keyword_scores[i]
        else:
            # Fetch from ChromaDB
            try:
                fetched = collection.get(ids=[doc_id], include=["documents", "metadatas"])
                if fetched["documents"]:
                    combined[doc_id] = {
                        "document": fetched["documents"][0],
                        "metadata": fetched["metadatas"][0],
                        "semantic_score": 0,
                        "keyword_score": keyword_scores[i]
                    }
            except:
                pass
    
    # Calculate hybrid scores
    for doc_id, data in combined.items():
        data["hybrid_score"] = alpha * data["semantic_score"] + (1 - alpha) * (data["keyword_score"] / (max(keyword_scores) if keyword_scores else 1))
    
    # Sort by hybrid score
    sorted_results = sorted(combined.items(), key=lambda x: x[1]["hybrid_score"], reverse=True)
    
    # Return top_k
    for doc_id, data in sorted_results[:top_k]:
        results["documents"].append(data["document"])
        results["metadatas"].append(data["metadata"])
        results["ids"].append(doc_id)
        results["distances"].append(1 - data["hybrid_score"])
    
    return results

In [ ]:
# ============================================================
# CELL 9: Test Retrieval
# ============================================================
test_queries = [
    "How do I make my first contribution?",
    "What are the steps to create a pull request?",
    "How do I run tests?",
    "What is the code style guide?"
]

In [ ]:
for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    results = hybrid_retrieve(query, top_k=3)
    
    for i, (doc, meta, dist) in enumerate(zip(results["documents"], results["metadatas"], results["distances"])):
        print(f"\n  Result {i+1} (score: {1-dist:.3f}):")
        print(f"    Source: {meta.get('source', 'unknown')}, Page: {meta.get('page', '?')}")
        print(f"    Text: {doc[:200]}...")

In [ ]:
# ============================================================
# CELL 10: RAG Generation with Gemma
# ============================================================
import re
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('models/gemma-4-26b-a4b-it')

def generate_answer(query: str, history: list = None, top_k: int = 5):
    """Generate answer using RAG pipeline."""
    
    # Retrieve relevant context
    results = hybrid_retrieve(query, top_k=top_k)
    
    if not results["documents"]:
        return {
            "answer": "I don't have enough information to answer that question. Please upload a contribution guide first.",
            "citations": [],
            "refused": True,
            "evidence": []
        }
    
    # Build context with short citation numbers
    context_parts = []
    citations = []
    chunk_ids = []
    
    for i, (doc, meta) in enumerate(zip(results["documents"], results["metadatas"])):
        source = meta.get("source", "unknown")
        page = meta.get("page", "?")
        chunk_id = f"chunk_{i}"
        chunk_ids.append(chunk_id)
        context_parts.append(f"[{i+1}] {doc}")
        citations.append({
            "source": source,
            "page": page,
            "text": doc[:200] + "..." if len(doc) > 200 else doc
        })
    
    context = "\n\n".join(context_parts)
    
    # Build conversation history
    history_text = ""
    if history:
        for h in history[-3:]:
            history_text += f"User: {h.get('q', '')}\nAssistant: {h.get('a', '')}\n\n"
    
    # Generate prompt with short citation numbers
    prompt = f"""Your response MUST start with "1." and contain ONLY numbered steps. Do NOT output any analysis, reasoning, goals, constraints, bullet points, or formatting markers.

Give clear numbered steps with brief practical details. Cite sources inline like [1], [2], etc.

Context:
{context}

Conversation History:
{history_text}
Question: {query}

1."""
    
    try:
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                max_output_tokens=300,
                temperature=0.1,
                top_p=0.8,
                top_k=20,
            )
        )
        
        # Post-process to extract only the actual answer (after Context:)
        text = response.text.strip()
        
        # Split by "Context:" and take the last part (the actual answer)
        parts = text.split('Context:')
        if len(parts) > 1:
            text = parts[-1]
        
        # Also split by "Question:" and take the last part
        parts = text.split('Question:')
        if len(parts) > 1:
            text = parts[-1]
        
        # Now extract numbered steps (both "1." and "Step 1:" formats) with citations
        lines = text.split('\n')
        step_lines = []
        
        for line in lines:
            stripped = line.strip()
            # Remove leading "*   " or similar bullet prefixes
            stripped = re.sub(r'^[\*\-\•\s]+', '', stripped)
            
            # Match both "1. text [cite]" and "Step 1: text [cite]" formats
            if (re.match(r'^\s*\d+[.)]\s+\w', stripped) or re.match(r'^\s*Step\s+\d+[:.)]\s+\w', stripped)) and re.search(r'\[\d+\]', stripped):
                step_lines.append(stripped)
            elif step_lines and stripped and not re.match(r'^\s*[\*\-\•]', stripped) and not re.match(r'^\s*(Constraint|Goal|Question|Context|Step\s*\d+:)', stripped):
                # Continuation of a step
                step_lines.append(stripped)
        
        if step_lines:
            answer = '\n'.join(step_lines[:8])
        else:
            # Fallback: find all numbered patterns with citations
            all_steps = re.findall(r'(?:^|\n)\s*(\d+[.)]\s*[^\n]*\[\d+\][^\n]*)', text)
            if all_steps:
                answer = '\n'.join(all_steps[-8:])
            else:
                # Also try Step format
                all_steps = re.findall(r'(?:^|\n)\s*(Step\s+\d+[:.)]\s*[^\n]*\[\d+\][^\n]*)', text)
                if all_steps:
                    answer = '\n'.join(all_steps[-8:])
                else:
                    answer = text
        
        # Check if answer indicates refusal
        refused = any(phrase in answer.lower() for phrase in [
            "don't know", "don't have", "cannot answer",
            "not enough information", "insufficient context"
])
        
        return {
            "answer": answer,
            "citations": citations,
            "refused": refused,
            "evidence": results["documents"]
        }
    except Exception as e:
        return {
            "answer": f"Error generating response: {str(e)}",
            "citations": [],
            "refused": True,
            "evidence": []
        }

In [ ]:
# ============================================================
# CELL 11: Test RAG Generation
# ============================================================
test_questions = [
    "How do I make my first contribution?",
    "What are the steps to create a pull request?",
    "How do I run tests?",
    "What is the code style guide?"
]

In [ ]:
chat_history = []

for question in test_questions:
    print(f"\n{'='*70}")
    print(f"Q: {question}")
    print(f"{'='*70}")
    
    result = generate_answer(question, history=chat_history)
    
    print(f"\nAnswer: {result['answer']}")
    print(f"Refused: {result['refused']}")
    print(f"Citations: {len(result['citations'])}")
    
    for i, cite in enumerate(result['citations']):
        print(f"  [{i+1}] {cite['source']} (page {cite['page']}): {cite['text'][:100]}...")
    
    chat_history.append({"q": question, "a": result['answer']})

In [ ]:
# ============================================================
# CELL 12: Onboarding / Getting Started Guide Generation
# ============================================================
def generate_onboarding():
    """Generate a getting started guide from indexed documents."""
    
    # Retrieve all documents for comprehensive guide
    all_docs = collection.get(include=["documents", "metadatas"])
    
    if not all_docs["documents"]:
        return {"steps": []}
    
    # Combine all content
    full_context = "\n\n".join(all_docs["documents"])
    
    prompt = f"""Based on the following contribution guide content, create a clear,
numbered step-by-step guide for a first-time contributor.
Each step should have a title and description.
Return as JSON array of objects with 'step', 'title', 'description'.

Content:
{full_context[:8000]}

JSON Response:"""
    
    try:
        response = model.generate_content(prompt)
        import json
        # Try to parse JSON from response
        text = response.text.strip()
        # Extract JSON if wrapped in markdown
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0]
        elif "```" in text:
            text = text.split("```")[1].split("```")[0]
        
        steps = json.loads(text)
        return {"steps": steps}
    except Exception as e:
        print(f"Error generating onboarding: {e}")
        # Fallback
        return {
            "steps": [
                {"step": 1, "title": "Fork the Repository", "description": "Click the Fork button on GitHub to create your own copy."},
                {"step": 2, "title": "Clone Locally", "description": "Clone your fork to your machine: git clone <your-fork-url>"},
                {"step": 3, "title": "Create Branch", "description": "Create a feature branch: git checkout -b feature/my-feature"},
                {"step": 4, "title": "Make Changes", "description": "Implement your feature or fix following the project's style guide."},
                {"step": 5, "title": "Submit PR", "description": "Push your branch and open a Pull Request with a clear description."}
            ]
        }

In [ ]:
# Test onboarding generation
onboarding = generate_onboarding()
print("Onboarding Steps:")
for step in onboarding["steps"]:
    print(f"  {step['step']}. {step['title']}: {step['description']}")

In [ ]:
# ============================================================
# CELL 13: Start FastAPI Server with ngrok Tunnel
# ============================================================
import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn
import threading
import time

# Kill any existing ngrok tunnels
ngrok.kill()

# Start ngrok tunnel on port 8000
public_url = ngrok.connect(8000, bind_tls=True)
print(f"🌐 Public API URL: {public_url}")
print(f"   Use this in your frontend: const API = '{public_url}'")

# Create FastAPI app inline for Colab
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, List
import shutil

app = FastAPI(title="ContribGuide AI - Colab Backend")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    question: str
    session_id: Optional[str] = None
    top_k: Optional[int] = 5
    mode: Optional[str] = "hybrid"
    threshold: Optional[float] = None

# In-memory session storage
sessions = {}

@app.get("/api/health")
def health():
    return {
        "status": "ok",
        "chunks": collection.count(),
        "model": "gemma-4-26b-a4b-it",
        "backend": "colab"
    }

@app.post("/api/chat")
def chat(req: ChatRequest):
    sid = req.session_id or "default"
    history = sessions.setdefault(sid, [])
    
    result = generate_answer(req.question, history=history, top_k=req.top_k)
    
    history.append({"q": req.question, "a": result["answer"]})
    sessions[sid] = history[-10:]  # Keep last 10
    
    return result

@app.post("/api/documents/upload")
async def upload_document(file: UploadFile = File(...)):
    ext = os.path.splitext(file.filename)[1].lower()
    if ext not in (".pdf", ".md", ".txt", ".markdown"):
        raise HTTPException(400, "Unsupported file type. Use PDF, MD, or TXT.")
    
    os.makedirs("/content/uploads", exist_ok=True)
    safe_name = f"{uuid.uuid4()}{ext}"
    filepath = os.path.join("/content/uploads", safe_name)
    
    with open(filepath, "wb") as f:
        shutil.copyfileobj(file.file, f)
    
    # Check file size
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    if size_mb > 20:
        os.remove(filepath)
        raise HTTPException(413, "File too large (max 20MB)")
    
    result = process_and_store_document(filepath, file.filename)
    return result

@app.get("/api/documents")
def list_documents():
    return {"total_chunks": collection.count()}

@app.post("/api/onboarding")
def onboarding():
    return generate_onboarding()

# Run server in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

# Test the API
import requests
try:
    response = requests.get(f"{public_url}/api/health", timeout=10)
    print(f"✓ Health check: {response.json()}")
except Exception as e:
    print(f"✗ Health check failed: {e}")

In [ ]:
# ============================================================
# CELL 14: Test Full API Endpoints
# ============================================================
import requests
import json

BASE_URL = str(public_url)

print(f"Testing API at: {BASE_URL}")
print("=" * 60)

# Test health
r = requests.get(f"{BASE_URL}/api/health")
print(f"Health: {r.json()}")

# Test chat
chat_payload = {
    "question": "How do I make my first contribution?",
    "top_k": 3
}
r = requests.post(f"{BASE_URL}/api/chat", json=chat_payload)
result = r.json()
print(f"\nChat Response:")
print(f"  Answer: {result['answer'][:300]}...")
print(f"  Citations: {len(result['citations'])}")
print(f"  Refused: {result['refused']}")

# Test onboarding
r = requests.post(f"{BASE_URL}/api/onboarding")
onboard = r.json()
print(f"\nOnboarding Steps: {len(onboard.get('steps', []))}")
for step in onboard.get('steps', [])[:3]:
    print(f"  {step['step']}. {step['title']}")

# ============================================================
# CELL 15: Connect Your Local Frontend
# ============================================================

## To connect your local frontend to this Colab backend:

1. **Copy the public URL above** (e.g., `https://abc123.ngrok-free.app`)

2. **Update your frontend** (`frontend/src/App.tsx`):
   ```typescript
   // Change this line:
   const API = "http://localhost:8000";
   
   // To your ngrok URL:
   const API = "https://your-ngrok-url.ngrok-free.app";
   ```

3. **Restart your frontend**:
   ```bash
   cd frontend
   npm run dev
   ```

4. **Test the full flow**:
   - Upload a document via the UI
   - Ask questions in the chat
   - Get AI responses with citations

## Important Notes:

- **ngrok URL changes** each session (unless you have a paid plan)
- **Colab times out** after 90 minutes of inactivity
- **Free GPU** (T4/K80) may be slower than local
- **For production**, deploy to Cloud Run, Vertex AI, or GKE

## Alternative: Run Frontend in Colab Too

You can also run the Vite frontend in Colab:
```bash
cd /content/contribguide-ai/frontend
npm install
npm run build
npx serve -s dist -l 3000
```
Then expose port 3000 with another ngrok tunnel.

In [ ]:
# ============================================================
# CELL 16: Upload Custom Document via API
# ============================================================
# You can upload files directly via the API
# Example using requests:

# import requests
# 
# with open("your-guide.pdf", "rb") as f:
#     files = {"file": ("your-guide.pdf", f, "application/pdf")}
#     response = requests.post(f"{public_url}/api/documents/upload", files=files)
#     print(response.json())

print("Ready to accept uploads via API at:")
print(f"  POST {public_url}/api/documents/upload")
print("  Content-Type: multipart/form-data")
print("  Field name: file")

# ============================================================
# Summary
# ============================================================

This notebook provides a complete RAG pipeline running on Google Colab:

✅ **Document Processing**: PDF, Markdown, TXT support
✅ **Chunking**: Overlapping chunks with metadata
✅ **Embeddings**: sentence-transformers/all-MiniLM-L6-v2
✅ **Vector Store**: ChromaDB with persistence
✅ **Hybrid Search**: Semantic + BM25 keyword search
✅ **Generation**: Google Gemma-4-26b-a4b-it via Generative AI
✅ **API Server**: FastAPI with CORS enabled
✅ **Public Access**: ngrok tunnel for frontend connectivity
✅ **Endpoints**: /api/health, /api/chat, /api/documents/upload, /api/onboarding

## Next Steps:
1. Replace `YOUR_GOOGLE_API_KEY_HERE` with your actual key
2. Replace `YOUR_USERNAME` with your GitHub username
3. Run all cells in order
4. Copy the ngrok URL and update your frontend
5. Test the complete flow!